# Pandas Notebook 04: Transformations and feature engineering

This notebook shows how to transform columns and create new features for analytics and beginner data science workflows.

## Notebook objectives

By the end, you should be able to create new columns, use vectorized operations, work with `.str` and `.dt`, use `map()`, `replace()`, `apply()`, `np.where()`, `cut()`, and `qcut()`, and build useful analysis-ready features.

### Transforming vs engineering

- **Transforming** changes an existing column into a more useful form.
- **Feature engineering** creates a new column that adds useful information for analysis.

We often do both together.

In [1]:
import pandas as pd
import numpy as np

## Dataset

We will use a local housing-style CSV stored in `sample_data`. It includes prices, sizes, dates, text, and categories, so it works well for feature engineering examples.

In [2]:
csv_path = "sample_data/housing_listings_sample.csv"
housing = pd.read_csv(csv_path)
housing["listing_date"] = pd.to_datetime(housing["listing_date"])

housing

,listing_id,city,neighborhood,property_type,price,size_sqft,bedrooms,bathrooms,listing_date,listing_title,energy_rating,furnished
0,1001,Bogota,Downtown,Apartment,120000,850,2,2,2025-01-10,Cozy Apartment,B,yes
1,1002,Bogota,North-End,House,245000,1600,3,3,2025-02-05,Family HOUSE,A,no
2,1003,Medellin,El Poblado,Apartment,180000,1100,2,2,2025-02-20,Modern Loft,B,yes
3,1004,Cali,San Fernando,Studio,90000,500,1,1,2025-03-12,budget studio,C,no
4,1005,Cali,Downtown,Apartment,130000,950,2,1,2025-03-25,Central Apartment,B,no
5,1006,Bogota,Chapinero,Apartment,150000,980,2,2,2025-04-10,Bright apartment,A,yes
6,1007,Medellin,Laureles,House,260000,1750,4,3,2025-04-22,Garden Home,A,no
7,1008,Barranquilla,Alto Prado,Apartment,140000,1020,3,2,2025-05-18,Sea-view condo,B,yes
8,1009,Cali,North-End,House,210000,1500,3,2,2025-06-03,Spacious Family Home,C,no
9,1010,Bogota,Usaquen,Apartment,195000,1200,3,2,2025-06-15,Work-From-Home Friendly,A,yes


## 1-3. Creating new columns, vectorized operations, and arithmetic between columns

Vectorized operations work on full columns at once. They are usually preferred over loops because they are simpler and faster.

These operations make it easy to create new columns such as ratios, totals, and scaled values.

In [3]:
housing_math = housing.copy()
housing_math["price_in_thousands"] = housing_math["price"] / 1000
housing_math["price_plus_5pct"] = housing_math["price"] * 1.05
housing_math["price_per_sqft"] = (housing_math["price"] / housing_math["size_sqft"]).round(2)
housing_math["total_rooms"] = housing_math["bedrooms"] + housing_math["bathrooms"]

housing_math[["listing_id", "price", "price_in_thousands", "price_plus_5pct", "size_sqft", "price_per_sqft", "total_rooms"]]

,listing_id,price,price_in_thousands,price_plus_5pct,size_sqft,price_per_sqft,total_rooms
0,1001,120000,120.0,126000.0,850,141.18,4
1,1002,245000,245.0,257250.0,1600,153.12,6
2,1003,180000,180.0,189000.0,1100,163.64,4
3,1004,90000,90.0,94500.0,500,180.00,2
4,1005,130000,130.0,136500.0,950,136.84,3
5,1006,150000,150.0,157500.0,980,153.06,4
6,1007,260000,260.0,273000.0,1750,148.57,7
7,1008,140000,140.0,147000.0,1020,137.25,5
8,1009,210000,210.0,220500.0,1500,140.00,5
9,1010,195000,195.0,204750.0,1200,162.50,5


## 4. String methods with `.str`

Use `.str` when you want to clean or transform text columns. This is great for titles, labels, city names, and categories.

In [4]:
housing_text = housing.copy()
housing_text["listing_title_clean"] = housing_text["listing_title"].str.strip().str.title()
housing_text["neighborhood_clean"] = housing_text["neighborhood"].str.replace("-", " ", regex=False).str.title()
housing_text["city_slug"] = housing_text["city"].str.lower().str.replace(" ", "_", regex=False)
housing_text["title_has_family"] = housing_text["listing_title"].str.strip().str.lower().str.contains("family")

housing_text[["listing_title", "listing_title_clean", "neighborhood", "neighborhood_clean", "city_slug", "title_has_family"]]

,listing_title,listing_title_clean,neighborhood,neighborhood_clean,city_slug,title_has_family
0,Cozy Apartment,Cozy Apartment,Downtown,Downtown,bogota,False
1,Family HOUSE,Family House,North-End,North End,bogota,True
2,Modern Loft,Modern Loft,El Poblado,El Poblado,medellin,False
3,budget studio,Budget Studio,San Fernando,San Fernando,cali,False
4,Central Apartment,Central Apartment,Downtown,Downtown,cali,False
5,Bright apartment,Bright Apartment,Chapinero,Chapinero,bogota,False
6,Garden Home,Garden Home,Laureles,Laureles,medellin,False
7,Sea-view condo,Sea-View Condo,Alto Prado,Alto Prado,barranquilla,False
8,Spacious Family Home,Spacious Family Home,North-End,North End,cali,True
9,Work-From-Home Friendly,Work-From-Home Friendly,Usaquen,Usaquen,bogota,False


## 5. Datetime accessors with `.dt`

After a column has been converted to datetime, `.dt` helps us pull out useful parts such as month, year, quarter, and weekday.

In [6]:
housing_dates = housing.copy()
housing_dates["listing_year"] = housing_dates["listing_date"].dt.year
housing_dates["listing_month"] = housing_dates["listing_date"].dt.month_name()
housing_dates["listing_quarter"] = housing_dates["listing_date"].dt.quarter
housing_dates["listing_weekday"] = housing_dates["listing_date"].dt.day_name()

housing_dates[["listing_id", "listing_date", "listing_year", "listing_month", "listing_quarter", "listing_weekday"]]

,listing_id,listing_date,listing_year,listing_month,listing_quarter,listing_weekday
0,1001,2025-01-10,2025,January,1,Friday
1,1002,2025-02-05,2025,February,1,Wednesday
2,1003,2025-02-20,2025,February,1,Thursday
3,1004,2025-03-12,2025,March,1,Wednesday
4,1005,2025-03-25,2025,March,1,Tuesday
5,1006,2025-04-10,2025,April,2,Thursday
6,1007,2025-04-22,2025,April,2,Tuesday
7,1008,2025-05-18,2025,May,2,Sunday
8,1009,2025-06-03,2025,June,2,Tuesday
9,1010,2025-06-15,2025,June,2,Sunday


## 6. `map()` and `replace()`

`map()` is helpful for one-to-one mappings, while `replace()` is useful when swapping specific values. Both are common in categorical feature engineering.

In [7]:
housing_map = housing.copy()
housing_map["energy_score"] = housing_map["energy_rating"].map({"A": 3, "B": 2, "C": 1})
housing_map["furnished_flag"] = housing_map["furnished"].map({"yes": 1, "no": 0})
housing_map["property_type_short"] = housing_map["property_type"].replace({"Apartment": "apt", "House": "house", "Studio": "studio"})

housing_map[["listing_id", "energy_rating", "energy_score", "furnished", "furnished_flag", "property_type", "property_type_short"]]

,listing_id,energy_rating,energy_score,furnished,furnished_flag,property_type,property_type_short
0,1001,B,2,yes,1,Apartment,apt
1,1002,A,3,no,0,House,house
2,1003,B,2,yes,1,Apartment,apt
3,1004,C,1,no,0,Studio,studio
4,1005,B,2,no,0,Apartment,apt
5,1006,A,3,yes,1,Apartment,apt
6,1007,A,3,no,0,House,house
7,1008,B,2,yes,1,Apartment,apt
8,1009,C,1,no,0,House,house
9,1010,A,3,yes,1,Apartment,apt


## 7-8. `apply()` and conditional feature creation with NumPy

`apply()` runs a function on each value or row. It is useful for custom logic, but if a vectorized option exists, that is usually the better first choice.

For simple conditional features, `np.where()` is often cleaner than row-by-row logic.

In [8]:
def home_size_label(sqft):
    if sqft >= 1400:
        return "large"
    elif sqft>= 900:
        return "medium"
    else: 
        return "small"

In [9]:
lambda sqft: "large" if sqft >= 1400 else "medium" if sqft >= 900 else "small"

<function __main__.<lambda>(sqft)>

In [12]:
housing_apply["size_sqft"]

0     850
1    1600
2    1100
3     500
4     950
5     980
6    1750
7    1020
8    1500
9    1200
Name: size_sqft, dtype: int64

In [13]:
housing_apply["size_sqft"].apply(home_size_label)

0     small
1     large
2    medium
3     small
4    medium
5    medium
6     large
7    medium
8     large
9    medium
Name: size_sqft, dtype: str

In [11]:
housing_apply = housing.copy()

# Vectorized solution
housing_apply["price_per_sqft_vectorized"] = (housing_apply["price"] / housing_apply["size_sqft"]).round(2)

# Same idea with apply
housing_apply["price_per_sqft_apply"] = housing_apply.apply(
    lambda fila: round(fila["price"] / fila["size_sqft"], 2),
    axis=1,
)

# A custom label with apply
housing_apply["home_size_label"] = housing_apply["size_sqft"].apply(
    lambda sqft: "large" if sqft >= 1400 else "medium" if sqft >= 900 else "small"
)

# A simple conditional feature with NumPy
housing_apply["is_family_home"] = np.where(housing_apply["bedrooms"] >= 3, "yes", "no")
housing_apply["listing_segment"] = np.where(
    (housing_apply["price"] >= 200000) | (housing_apply["size_sqft"] >= 1500),
    "premium",
    "standard",
)

housing_apply[["listing_id", "price_per_sqft_vectorized", "price_per_sqft_apply", "home_size_label", "is_family_home", "listing_segment"]]

,listing_id,price_per_sqft_vectorized,price_per_sqft_apply,home_size_label,is_family_home,listing_segment
0,1001,141.18,141.18,small,no,standard
1,1002,153.12,153.12,large,yes,premium
2,1003,163.64,163.64,medium,no,standard
3,1004,180.00,180.00,small,no,standard
4,1005,136.84,136.84,medium,no,standard
5,1006,153.06,153.06,medium,no,standard
6,1007,148.57,148.57,large,yes,premium
7,1008,137.25,137.25,medium,yes,standard
8,1009,140.00,140.00,large,yes,premium
9,1010,162.50,162.50,medium,yes,standard


## 9. Binning variables with `cut()` and `qcut()`

`cut()` uses bins you define. `qcut()` creates groups with roughly similar numbers of rows. Both are useful for turning continuous numbers into categories.

In [15]:
housing_bins = housing.copy()
housing_bins["size_band"] = pd.cut(housing_bins["size_sqft"], bins=[0, 700, 1200, 2000], labels=["small", "medium", "large"])
housing_bins["price_tier"] = pd.qcut(housing_bins["price"], q=3, labels=["budget", "mid", "premium"])

housing_bins[["listing_id", "price", "size_sqft", "size_band", "price_tier"]]

,listing_id,price,size_sqft,size_band,price_tier
0,1001,120000,850,medium,budget
1,1002,245000,1600,large,premium
2,1003,180000,1100,medium,mid
3,1004,90000,500,small,budget
4,1005,130000,950,medium,budget
5,1006,150000,980,medium,mid
6,1007,260000,1750,large,premium
7,1008,140000,1020,medium,budget
8,1009,210000,1500,large,premium
9,1010,195000,1200,medium,mid


## 10. Useful feature engineering examples for analysis

A good feature should help answer a question more easily. In housing data, helpful features often describe value, time, category, or household suitability.

In [17]:
features_df = housing.copy()
features_df["listing_title_clean"] = features_df["listing_title"].str.strip().str.title()
features_df["price_per_sqft"] = (features_df["price"] / features_df["size_sqft"]).round(2)
features_df["listing_month"] = features_df["listing_date"].dt.month_name()
features_df["energy_score"] = features_df["energy_rating"].map({"A": 3, "B": 2, "C": 1})
features_df["furnished_flag"] = features_df["furnished"].map({"yes": 1, "no": 0})
features_df["is_family_home"] = np.where(features_df["bedrooms"] >= 3, 1, 0)
features_df["price_tier"] = pd.qcut(features_df["price"], q=3, labels=["budget", "mid", "premium"])
features_df["value_label"] = np.where(features_df["price_per_sqft"] < features_df["price_per_sqft"].median(), "better_value", "higher_cost")

features_df[["listing_id", "city", "property_type", "price_per_sqft", "listing_month", "energy_score", "furnished_flag", "is_family_home", "price_tier", "value_label"]]

,listing_id,city,property_type,price_per_sqft,listing_month,energy_score,furnished_flag,is_family_home,price_tier,value_label
0,1001,Bogota,Apartment,141.18,January,2,1,0,budget,better_value
1,1002,Bogota,House,153.12,February,3,0,1,premium,higher_cost
2,1003,Medellin,Apartment,163.64,February,2,1,0,mid,higher_cost
3,1004,Cali,Studio,180.00,March,1,0,0,budget,higher_cost
4,1005,Cali,Apartment,136.84,March,2,0,0,budget,better_value
5,1006,Bogota,Apartment,153.06,April,3,1,0,mid,higher_cost
6,1007,Medellin,House,148.57,April,3,0,1,premium,better_value
7,1008,Barranquilla,Apartment,137.25,May,2,1,1,budget,better_value
8,1009,Cali,House,140.00,June,1,0,1,premium,better_value
9,1010,Bogota,Apartment,162.50,June,3,1,1,mid,higher_cost


## Practice exercises

Try these with `housing`.

1. Create a `price_per_bedroom` column.
2. Create a cleaned version of `listing_title`.
3. Create a `listing_month_number` column from `listing_date`.
4. Map `energy_rating` to a numeric score.
5. Use `np.where()` to label listings as `"large"` or `"not_large"` based on `size_sqft`.
6. Use `cut()` or `qcut()` to create a grouped price column.

In [33]:
# Write your practice answers here.
housing_practice = housing.copy()
housing_practice["price_per_bedroom"] = (housing_practice["price"] / housing_practice["bedrooms"]).round(2)
housing_practice["listing_title_cleaned"] = housing_practice["listing_title"].str.strip().str.replace("-", " ", regex=False).str.title()
housing_practice["listing_month_number"] = housing_practice["listing_date"].dt.month
housing_practice["energy_score"] = housing_practice["energy_rating"].map({"A": 100, "B": 70, "C": 30})
housing_practice["is_large"] = np.where(housing_practice["size_sqft"] < housing_practice["size_sqft"].median(), "large", "not large")
housing_practice["price_tier"] = pd.qcut(housing_practice["price"], q=3, labels=["budget", "mid", "premium"])
housing_practice[["price_per_bedroom", "listing_title_cleaned", "listing_month_number", "is_large", "price_tier"]]

,price_per_bedroom,listing_title_cleaned,listing_month_number,is_large,price_tier
0,60000.00,Cozy Apartment,1,large,budget
1,81666.67,Family House,2,not large,premium
2,90000.00,Modern Loft,2,not large,mid
3,90000.00,Budget Studio,3,large,budget
4,65000.00,Central Apartment,3,large,budget
5,75000.00,Bright Apartment,4,large,mid
6,65000.00,Garden Home,4,not large,premium
7,46666.67,Sea View Condo,5,large,budget
8,70000.00,Spacious Family Home,6,not large,premium
9,65000.00,Work From Home Friendly,6,not large,mid


## Mini challenge

Prepare a new DataFrame for later analysis.

Create at least these features:

- `price_per_sqft`
- `listing_month`
- `energy_score`
- `is_family_home`
- `price_tier`
- one extra feature of your own choice

Then answer:

- Which listings are in the `premium` tier?
- Which listings look like better value?
- Which city seems to have more family-sized homes in this sample?

In [35]:
# Write your mini challenge solution here.
mini_challenge = housing.copy()
mini_challenge.head()

,listing_id,city,neighborhood,property_type,price,size_sqft,bedrooms,bathrooms,listing_date,listing_title,energy_rating,furnished
0,1001,Bogota,Downtown,Apartment,120000,850,2,2,2025-01-10,Cozy Apartment,B,yes
1,1002,Bogota,North-End,House,245000,1600,3,3,2025-02-05,Family HOUSE,A,no
2,1003,Medellin,El Poblado,Apartment,180000,1100,2,2,2025-02-20,Modern Loft,B,yes
3,1004,Cali,San Fernando,Studio,90000,500,1,1,2025-03-12,budget studio,C,no
4,1005,Cali,Downtown,Apartment,130000,950,2,1,2025-03-25,Central Apartment,B,no


In [62]:
mini_challenge["listing_title_cleaned"] = housing_practice["listing_title"].str.strip().str.replace("-", " ", regex=False).str.title()
mini_challenge["price_per_sqft"] = (housing_practice["price"] / housing_practice["bedrooms"]).round(2)
mini_challenge["listing_month"] = housing_practice["listing_date"].dt.month_name()
mini_challenge["energy_score"] = mini_challenge["energy_rating"].map({"A": 100, "B": 70, "C": 30})
mini_challenge["is_family_home"] = np.where(mini_challenge["bedrooms"] >= 3, "yes", "no")
mini_challenge["price_tier"] = pd.qcut(mini_challenge["price"], q=3, labels=["budget", "mid", "premium"])
mini_challenge["sqft_per_bedroom"] = (housing_practice["size_sqft"] / housing_practice["bedrooms"]).round(2)
mini_challenge[["price_per_sqft", "listing_month", "energy_score", "is_family_home", "price_tier", "sqft_per_bedroom"]]

,price_per_sqft,listing_month,energy_score,is_family_home,price_tier,sqft_per_bedroom
0,60000.00,January,70,no,budget,425.00
1,81666.67,February,100,yes,premium,533.33
2,90000.00,February,70,no,mid,550.00
3,90000.00,March,30,no,budget,500.00
4,65000.00,March,70,no,budget,475.00
5,75000.00,April,100,no,mid,490.00
6,65000.00,April,100,yes,premium,437.50
7,46666.67,May,70,yes,budget,340.00
8,70000.00,June,30,yes,premium,500.00
9,65000.00,June,100,yes,mid,400.00


In [66]:
# Which listings are in the premium tier?
premium_listings = mini_challenge[mini_challenge["price_tier"] == "premium"]
print("The premium listings are:\n",premium_listings[["listing_id", "listing_title_cleaned", "price", "price_tier"]])

# Which listings look like better value?
better_value = mini_challenge[mini_challenge["price_per_sqft"] > mini_challenge["price_per_sqft"].mean()]
print("The better value listings are:\n",better_value[["listing_id", "listing_title_cleaned", "price_per_sqft","price"]])

# Which city seems to have more family-sized homes in this sample?
family_sized_homes = mini_challenge[mini_challenge["is_family_home"] == "yes"]
print("The family-sized homes are:\n", family_sized_homes[["listing_id","listing_title_cleaned","is_family_home","bedrooms","price"]])

The premium listings are:
    listing_id listing_title_cleaned   price price_tier
1        1002          Family House  245000    premium
6        1007           Garden Home  260000    premium
8        1009  Spacious Family Home  210000    premium
The better value listings are:
    listing_id listing_title_cleaned  price_per_sqft   price
1        1002          Family House        81666.67  245000
2        1003           Modern Loft        90000.00  180000
3        1004         Budget Studio        90000.00   90000
5        1006      Bright Apartment        75000.00  150000
The family-sized homes are:
    listing_id    listing_title_cleaned is_family_home  bedrooms   price
1        1002             Family House            yes         3  245000
6        1007              Garden Home            yes         4  260000
7        1008           Sea View Condo            yes         3  140000
8        1009     Spacious Family Home            yes         3  210000
9        1010  Work From Home Fr

## When to use each method

- Use column assignment for straightforward new columns.
- Use vectorized arithmetic for fast column-wide math.
- Use `.str`} for text work.
- Use `.dt` for date parts after datetime conversion.
- Use `map()` for one-to-one category mappings.
- Use `replace()` for targeted value swaps.
- Use `apply()` for custom logic when vectorized methods are not a good fit.
- Use `np.where()` for simple conditional labels.
- Use `cut()` for custom bins and `qcut()` for roughly equal-sized groups.

## Key takeaways

- transformations improve existing columns, while feature engineering creates new analytical columns
- vectorized methods are usually the best first choice in pandas
- text, dates, mappings, arithmetic, and conditions all help create useful features
- `apply()` is helpful, but it should not replace simpler vectorized solutions
- binned features can make reporting and analysis easier